# Part 6 — Weekly SKU-Level Forecasting

**Goal:** Give Supply Chain a SKU-level forecast so they know what to stock,
what to reorder, and which SKUs are at risk of stocking out during the promo.

---

## Data Honesty Statement

`orders_clean.csv` has order-level SKU lists (pipe-delimited) and total units
per order — but no line-item breakdown of units per SKU. We approximate:

> **units per SKU = order units ÷ number of SKUs in that order**

This assumption is documented explicitly. Orders where one SKU accounts for
multiple units will be slightly mis-attributed, but the aggregate error across
hundreds of orders is small and directionally correct.

---

## Notebook Structure

1. Load all data files
2. Explode orders to SKU level — derive TY baseline per SKU
3. Load LY promo data — compute per-SKU uplift factors
4. Tier SKUs by uplift magnitude — justify tier assignments
5. Forecast promo week units per SKU (low / mid / high)
6. Forecast post-promo week per SKU (pull-forward hangover)
7. Roll up: total units, revenue, SKU contribution %
8. Stockout risk analysis — compare forecast to current inventory
9. Supply chain brief — 3 bullets


## 1. Imports & Constants

In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

BASELINE_START = pd.Timestamp("2025-02-28")
BASELINE_END   = pd.Timestamp("2025-04-24")
PROMO_START    = pd.Timestamp("2025-04-25")
PROMO_END      = pd.Timestamp("2025-04-27")
PROMO_CODE     = "SPRING20"
BASELINE_DAYS  = 56
PROMO_DAYS     = 3

# YoY growth and uplift scalars from Part 5
GROWTH = {"low": 1.10, "mid": 1.18, "high": 1.25}
UPLIFT_SCALAR = {"low": 0.85, "mid": 1.00, "high": 1.10}


## 2. Load Data

In [7]:
orders  = pd.read_csv("/Users/apple/Desktop/Assignment2/Output/orders_clean.csv", parse_dates=["order_date"])
ly      = pd.read_csv("/Users/apple/Desktop/Assignment2/Source Data/last_year_promo_skus.csv")
ty_skus = pd.read_csv("/Users/apple/Desktop/Assignment2/Source Data/this_year_promo_skus.csv")
skus    = pd.read_csv("/Users/apple/Desktop/Assignment2/Source Data/skus.csv")

# Normalise columns
for df in [ly, ty_skus, skus]:
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print(f"orders     : {len(orders):,} rows")
print(f"ly         : {len(ly)} rows  | phases: {sorted(ly['phase'].unique())}")
print(f"ty_skus    : {len(ty_skus)} rows")
print(f"skus       : {len(skus)} rows")
print()
print("=== THIS YEAR PROMO SKUs ===")
print(ty_skus)
print()
print("=== SKU MASTER ===")
print(skus)


orders     : 5,678 rows
ly         : 40 rows  | phases: ['Post-promo', 'Pre-promo', 'Promo week']
ty_skus    : 10 rows
skus       : 10 rows

=== THIS YEAR PROMO SKUs ===
          sku in_promo  current_inventory_units
0  CYM-GLU-30        Y                      499
1  CYM-VTC-30        Y                      233
2  CYM-VTD-30        Y                      261
3  CYM-MAG-30        Y                      315
4  CYM-OMG-60        Y                      218
5  CYM-MTB-30        Y                      273
6  CYM-CRE-30        N                      135
7  CYM-PRO-30        Y                      456
8  CYM-ASH-30        Y                      458
9  CYM-COL-30        Y                      211

=== SKU MASTER ===
          sku              product_name     category  unit_price
0  CYM-GLU-30     Liposomal Glutathione  Antioxidant       78.00
1  CYM-VTC-30       Liposomal Vitamin C     Immunity       52.00
2  CYM-VTD-30           Vitamin D3 + K2     Immunity       48.00
3  CYM-MAG-30     Magn

## 3. Explode Orders to SKU Level

**DECISION:** Units per SKU = total order units ÷ number of SKUs in that order.

Rationale: no line-item data available. Looking at the data, most orders have
the same count of units and SKUs, suggesting 1 unit per SKU is the dominant
pattern. Equal-split is the most defensible approximation without line-item data.


In [8]:
# Explode the pipe-delimited skus column
orders_exp = orders.copy()
orders_exp["sku_list"] = orders_exp["skus"].str.split("|")
orders_exp["n_skus"]   = orders_exp["sku_list"].apply(len)

# Explode: one row per SKU per order
orders_exp = orders_exp.explode("sku_list").rename(columns={"sku_list": "sku"})
orders_exp["sku"] = orders_exp["sku"].str.strip()

# Approximate units and revenue per SKU
orders_exp["sku_units"]   = orders_exp["units"]     / orders_exp["n_skus"]
orders_exp["sku_revenue"] = orders_exp["net_sales"]  / orders_exp["n_skus"]

print(f"Exploded orders: {len(orders_exp):,} SKU-order rows")
print(f"Unique SKUs in orders: {orders_exp['sku'].nunique()}")
print()

# Sanity check: unit totals should match original
orig_units = orders["units"].sum()
expl_units = orders_exp["sku_units"].sum()
print(f"Unit total check — original: {orig_units:,.1f}  exploded: {expl_units:,.1f}  diff: {abs(orig_units-expl_units):.4f}")


Exploded orders: 11,702 SKU-order rows
Unique SKUs in orders: 10

Unit total check — original: 11,702.0  exploded: 11,702.0  diff: 0.0000


## 4. TY Baseline Per SKU

Compute average daily units and revenue per SKU over the 8-week baseline window.
This is the "normal" rate for each SKU before any promo effect.


In [9]:
baseline_exp = orders_exp[
    (orders_exp["order_date"] >= BASELINE_START) &
    (orders_exp["order_date"] <= BASELINE_END)
]

ty_baseline_sku = (
    baseline_exp
    .groupby("sku")
    .agg(
        baseline_total_units   = ("sku_units",   "sum"),
        baseline_total_revenue = ("sku_revenue", "sum"),
    )
    .reset_index()
)

ty_baseline_sku["baseline_daily_units"]   = ty_baseline_sku["baseline_total_units"]   / BASELINE_DAYS
ty_baseline_sku["baseline_daily_revenue"] = ty_baseline_sku["baseline_total_revenue"] / BASELINE_DAYS
ty_baseline_sku["baseline_weekly_units"]  = ty_baseline_sku["baseline_daily_units"]   * 7

print("=== TY BASELINE PER SKU (daily avg) ===")
print(ty_baseline_sku.sort_values("baseline_daily_units", ascending=False).to_string(index=False))


=== TY BASELINE PER SKU (daily avg) ===
       sku  baseline_total_units  baseline_total_revenue  baseline_daily_units  baseline_daily_revenue  baseline_weekly_units
CYM-VTC-30                879.00               48,319.20                 15.70                  862.84                 109.88
CYM-GLU-30                834.00               55,346.63                 14.89                  988.33                 104.25
CYM-MAG-30                824.00               49,083.58                 14.71                  876.49                 103.00
CYM-ASH-30                820.00               43,939.52                 14.64                  784.63                 102.50
CYM-CRE-30                806.00               41,584.21                 14.39                  742.58                 100.75
CYM-VTD-30                803.00               42,923.28                 14.34                  766.49                 100.38
CYM-MTB-30                796.00               45,852.15                 14.21

## 5. LY Per-SKU Uplift Factors

From `last_year_promo_skus.csv` we compute:
- **LY pre-promo avg** = mean of W15 and W16 units
- **LY uplift factor** = promo week units ÷ pre-promo avg weekly units

This tells us how strongly each SKU responded to the promo last year.
SKUs with high uplift (>2.5x) are likely the bestsellers that customers
specifically wait for sales to buy. SKUs with low uplift (<1.8x) may be
staples that customers buy regardless.


In [10]:
ly_pre   = ly[ly["phase"] == "Pre-promo"].groupby("sku")["units_sold"].mean().reset_index()
ly_promo = ly[ly["phase"] == "Promo week"][["sku","units_sold"]].copy()
ly_post  = ly[ly["phase"] == "Post-promo"][["sku","units_sold"]].copy()

ly_pre.columns   = ["sku", "ly_pre_avg_weekly"]
ly_promo.columns = ["sku", "ly_promo_weekly"]
ly_post.columns  = ["sku", "ly_post_weekly"]

ly_sku = ly_pre.merge(ly_promo, on="sku").merge(ly_post, on="sku")
ly_sku["ly_uplift_factor"]    = ly_sku["ly_promo_weekly"] / ly_sku["ly_pre_avg_weekly"]
ly_sku["ly_post_ratio"]       = ly_sku["ly_post_weekly"]  / ly_sku["ly_pre_avg_weekly"]

print("=== LY PER-SKU UPLIFT FACTORS ===")
print(ly_sku.sort_values("ly_uplift_factor", ascending=False).to_string(index=False))
print()
print(f"Uplift range: {ly_sku['ly_uplift_factor'].min():.2f}x – {ly_sku['ly_uplift_factor'].max():.2f}x")
print(f"Uplift mean : {ly_sku['ly_uplift_factor'].mean():.2f}x")
print(f"Post ratio range: {ly_sku['ly_post_ratio'].min():.2f}x – {ly_sku['ly_post_ratio'].max():.2f}x")


=== LY PER-SKU UPLIFT FACTORS ===
       sku  ly_pre_avg_weekly  ly_promo_weekly  ly_post_weekly  ly_uplift_factor  ly_post_ratio
CYM-COL-30              81.50              255              66              3.13           0.81
CYM-ASH-30             124.00              380             102              3.06           0.82
CYM-GLU-30             117.00              357              90              3.05           0.77
CYM-PRO-30             113.50              340              91              3.00           0.80
CYM-OMG-60              70.50              205              54              2.91           0.77
CYM-VTC-30              88.50              248              83              2.80           0.94
CYM-MTB-30              61.00              163              57              2.67           0.93
CYM-VTD-30              86.50              231              66              2.67           0.76
CYM-MAG-30             141.50              367             120              2.59           0.85
CYM-CR

## 6. SKU Tiering by Uplift Magnitude

Rather than per-SKU growth multipliers (which would be noisy given only 1 year
of LY data), we group SKUs into three tiers and apply tier-level adjustments.

**Tier definitions:**

| Tier | LY uplift | Interpretation | Growth adj |
|---|---|---|---|
| High | ≥ 2.5x | Hero SKUs — customers specifically wait for sales | LY factor × 1.05 |
| Mid  | 1.8x – 2.5x | Strong performers — good promo response | LY factor × 1.00 |
| Low  | < 1.8x | Staples — bought regardless of promo | LY factor × 0.95 |

**Rationale:** Hero SKUs (high tier) are likely to grow faster as brand awareness
increases — more customers discover them each year. Staples (low tier) are
already at near-saturation for promo-driven demand; applying a slight haircut
is conservative and appropriate.


In [11]:
def assign_tier(uplift):
    if uplift >= 2.5:
        return "High"
    elif uplift >= 1.8:
        return "Mid"
    else:
        return "Low"

def tier_growth_adj(tier):
    return {"High": 1.05, "Mid": 1.00, "Low": 0.95}[tier]

ly_sku["tier"]            = ly_sku["ly_uplift_factor"].apply(assign_tier)
ly_sku["tier_growth_adj"] = ly_sku["tier"].apply(tier_growth_adj)

print("=== SKU TIER ASSIGNMENTS ===")
print(ly_sku[["sku","ly_uplift_factor","tier","tier_growth_adj",
              "ly_post_ratio"]].sort_values("ly_uplift_factor", ascending=False).to_string(index=False))
print()
print("Tier distribution:")
print(ly_sku["tier"].value_counts())


=== SKU TIER ASSIGNMENTS ===
       sku  ly_uplift_factor tier  tier_growth_adj  ly_post_ratio
CYM-COL-30              3.13 High             1.05           0.81
CYM-ASH-30              3.06 High             1.05           0.82
CYM-GLU-30              3.05 High             1.05           0.77
CYM-PRO-30              3.00 High             1.05           0.80
CYM-OMG-60              2.91 High             1.05           0.77
CYM-VTC-30              2.80 High             1.05           0.94
CYM-MTB-30              2.67 High             1.05           0.93
CYM-VTD-30              2.67 High             1.05           0.76
CYM-MAG-30              2.59 High             1.05           0.85
CYM-CRE-30              2.49  Mid             1.00           0.81

Tier distribution:
tier
High    9
Mid     1
Name: count, dtype: int64


## 7. Build Master SKU Table

In [12]:
# Join all SKU data together
sku_master = (
    ty_skus
    .merge(skus,           on="sku", how="left")
    .merge(ly_sku,         on="sku", how="left")
    .merge(ty_baseline_sku[["sku","baseline_daily_units","baseline_weekly_units",
                             "baseline_daily_revenue"]], on="sku", how="left")
)

# Fill SKUs not in LY data with mean uplift (shouldn't happen with this dataset)
mean_uplift = ly_sku["ly_uplift_factor"].mean()
mean_post   = ly_sku["ly_post_ratio"].mean()
sku_master["ly_uplift_factor"] = sku_master["ly_uplift_factor"].fillna(mean_uplift)
sku_master["ly_post_ratio"]    = sku_master["ly_post_ratio"].fillna(mean_post)
sku_master["tier"]             = sku_master["tier"].fillna("Mid")
sku_master["tier_growth_adj"]  = sku_master["tier_growth_adj"].fillna(1.00)

# Only keep SKUs that are IN this year's promo
sku_master = sku_master[sku_master["in_promo"] == "Y"].reset_index(drop=True)

print(f"SKUs in this year's promo: {len(sku_master)}")
print()
cols = ["sku","product_name","category","unit_price",
        "current_inventory_units","tier","ly_uplift_factor",
        "baseline_weekly_units","ly_post_ratio"]
print(sku_master[cols].to_string(index=False))


SKUs in this year's promo: 9

       sku             product_name    category  unit_price  current_inventory_units tier  ly_uplift_factor  baseline_weekly_units  ly_post_ratio
CYM-GLU-30    Liposomal Glutathione Antioxidant       78.00                      499 High              3.05                 104.25           0.77
CYM-VTC-30      Liposomal Vitamin C    Immunity       52.00                      233 High              2.80                 109.88           0.94
CYM-VTD-30          Vitamin D3 + K2    Immunity       48.00                      261 High              2.67                 100.38           0.76
CYM-MAG-30    Magnesium L-Threonate       Sleep       62.00                      315 High              2.59                 103.00           0.85
CYM-OMG-60              Omega-3 DHA       Heart       55.00                      218 High              2.91                  99.12           0.77
CYM-MTB-30     Methylated B Complex      Energy       58.00                      273 High     

## 8. Promo Week Forecast Per SKU

### Formula

```
TY promo weekly units (SKU) =
    LY promo weekly units (SKU)
    × YoY growth multiplier
    × tier growth adjustment
    × uplift scalar (scenario)
```

We convert LY weekly units to TY-equivalent by scaling using the same
0.75 factor from Part 5 (TY was ~133% of LY, so LY × 1/0.75 = LY × 1.333).

Then apply growth, tier adjustment, and scenario uplift scalar.


In [13]:
# Convert LY promo units to TY-equivalent baseline
# LY promo = 75% of TY promo → TY-equivalent LY = LY / 0.75
sku_master["ly_promo_weekly_ty_equiv"] = sku_master["ly_promo_weekly"] / 0.75

forecast_rows = []

for _, row in sku_master.iterrows():
    for scenario in ["low", "mid", "high"]:
        g  = GROWTH[scenario]
        u  = UPLIFT_SCALAR[scenario]
        ta = row["tier_growth_adj"]

        # Promo week
        proj_weekly = row["ly_promo_weekly_ty_equiv"] * g * ta * u

        # Post-promo week (demand hangover)
        # Post ratio from LY, dampened slightly by growth (larger customer base
        # means post-promo dip is relatively smaller)
        post_weekly = row["baseline_weekly_units"] * row["ly_post_ratio"] * g * 0.95

        # Revenue (use unit_price as proxy; actual promo revenue = price × 0.80)
        promo_revenue = proj_weekly * row["unit_price"] * 0.80
        post_revenue  = post_weekly * row["unit_price"]

        forecast_rows.append({
            "sku"              : row["sku"],
            "product_name"     : row["product_name"],
            "category"         : row["category"],
            "unit_price"       : row["unit_price"],
            "tier"             : row["tier"],
            "current_inventory": row["current_inventory_units"],
            "baseline_weekly"  : round(row["baseline_weekly_units"], 1),
            "scenario"         : scenario,
            "promo_weekly_units": round(proj_weekly, 1),
            "post_weekly_units" : round(post_weekly, 1),
            "total_2wk_units"  : round(proj_weekly + post_weekly, 1),
            "promo_revenue"    : round(promo_revenue, 2),
            "post_revenue"     : round(post_revenue, 2),
            "total_2wk_revenue": round(promo_revenue + post_revenue, 2),
        })

fc = pd.DataFrame(forecast_rows)

print("=== PROMO WEEK FORECAST — MID SCENARIO ===")
mid_fc = fc[fc["scenario"] == "mid"].copy()
print(mid_fc[[
    "sku","tier","baseline_weekly","promo_weekly_units","post_weekly_units",
    "total_2wk_units","promo_revenue"
]].sort_values("promo_weekly_units", ascending=False).to_string(index=False))


=== PROMO WEEK FORECAST — MID SCENARIO ===
       sku tier  baseline_weekly  promo_weekly_units  post_weekly_units  total_2wk_units  promo_revenue
CYM-ASH-30 High           102.50              627.80              94.50           722.30      24,608.19
CYM-MAG-30 High           103.00              606.30              97.90           704.20      30,071.69
CYM-GLU-30 High           104.20              589.80              89.90           679.70      36,801.27
CYM-PRO-30 High            98.50              561.70              88.50           650.20      29,207.36
CYM-COL-30 High            98.10              421.30              89.10           510.30      22,916.54
CYM-VTC-30 High           109.90              409.70             115.50           525.20      17,043.35
CYM-VTD-30 High           100.40              381.60              85.90           467.50      14,653.90
CYM-OMG-60 High            99.10              338.70              85.10           423.80      14,901.04
CYM-MTB-30 High      

## 9. Low / Mid / High Range Per SKU

In [14]:
pivot = fc.pivot_table(
    index=["sku","product_name","tier","current_inventory"],
    columns="scenario",
    values=["promo_weekly_units","total_2wk_units","promo_revenue"]
).round(1)

pivot.columns = [f"{col[0]}_{col[1]}" for col in pivot.columns]
pivot = pivot.reset_index().sort_values("promo_weekly_units_mid", ascending=False)

print("=== PROMO WEEK UNITS: LOW / MID / HIGH PER SKU ===")
display_cols = ["sku","tier","current_inventory",
                "promo_weekly_units_low","promo_weekly_units_mid","promo_weekly_units_high",
                "total_2wk_units_low","total_2wk_units_mid","total_2wk_units_high"]
print(pivot[display_cols].to_string(index=False))


=== PROMO WEEK UNITS: LOW / MID / HIGH PER SKU ===
       sku tier  current_inventory  promo_weekly_units_low  promo_weekly_units_mid  promo_weekly_units_high  total_2wk_units_low  total_2wk_units_mid  total_2wk_units_high
CYM-ASH-30 High                458                  497.40                  627.80                   731.50               585.50               722.30                831.60
CYM-MAG-30 High                315                  480.40                  606.30                   706.50               571.70               704.20                810.20
CYM-GLU-30 High                499                  467.30                  589.80                   687.20               551.10               679.70                782.50
CYM-PRO-30 High                456                  445.10                  561.70                   654.50               527.60               650.20                748.30
CYM-COL-30 High                211                  333.80                  421.30       

## 10. Roll-Up: Total Units, Revenue, SKU Contribution

Aggregate across all in-promo SKUs to get the total picture Supply Chain needs.
Also compute each SKU's share of total promo revenue (contribution %).


In [15]:
# Total roll-up by scenario
rollup = (
    fc.groupby("scenario")
    .agg(
        total_promo_units   = ("promo_weekly_units", "sum"),
        total_post_units    = ("post_weekly_units",  "sum"),
        total_2wk_units     = ("total_2wk_units",    "sum"),
        total_promo_revenue = ("promo_revenue",       "sum"),
        total_post_revenue  = ("post_revenue",        "sum"),
        total_2wk_revenue   = ("total_2wk_revenue",   "sum"),
    )
    .reset_index()
)

print("=== ROLL-UP: ALL IN-PROMO SKUs ===")
print(rollup.set_index("scenario").round(1).to_string())
print()

# SKU contribution to promo revenue (mid scenario)
mid_total_rev = fc[fc["scenario"]=="mid"]["promo_revenue"].sum()
mid_contrib = (
    fc[fc["scenario"]=="mid"][["sku","product_name","category","tier","promo_revenue"]]
    .copy()
)
mid_contrib["contribution_pct"] = (mid_contrib["promo_revenue"] / mid_total_rev * 100).round(1)
mid_contrib = mid_contrib.sort_values("contribution_pct", ascending=False)

print("=== SKU CONTRIBUTION TO PROMO REVENUE (MID) ===")
print(mid_contrib[["sku","category","tier","promo_revenue","contribution_pct"]].to_string(index=False))
print()
print(f"Top 3 SKUs account for "
      f"{mid_contrib['contribution_pct'].head(3).sum():.1f}% of promo revenue.")


=== ROLL-UP: ALL IN-PROMO SKUs ===
          total_promo_units  total_post_units  total_2wk_units  total_promo_revenue  total_post_revenue  total_2wk_revenue
scenario                                                                                                                  
high               4,901.10            901.10         5,802.20           236,194.40           53,368.60         289,563.00
low                3,332.70            792.90         4,125.60           160,612.20           46,964.30         207,576.50
mid                4,206.20            850.60         5,056.70           202,697.80           50,379.90         253,077.70

=== SKU CONTRIBUTION TO PROMO REVENUE (MID) ===
       sku    category tier  promo_revenue  contribution_pct
CYM-GLU-30 Antioxidant High      36,801.27             18.20
CYM-MAG-30       Sleep High      30,071.69             14.80
CYM-PRO-30         Gut High      29,207.36             14.40
CYM-ASH-30      Stress High      24,608.19             12

## 11. Stockout Risk Analysis

Compare forecasted 2-week units (promo + post-promo) against `current_inventory_units`.

**Risk flag logic:**
- **Critical:** current inventory < low scenario forecast → will stock out even in the most conservative case
- **At risk:** current inventory < mid scenario forecast → stocks out at expected performance
- **Watch:** current inventory < high scenario forecast → only stocks out if promo outperforms
- **OK:** current inventory ≥ high scenario forecast → sufficient for all scenarios


In [16]:
# Get 2-week forecast per scenario per SKU
risk = pivot[["sku","product_name","tier","current_inventory",
              "total_2wk_units_low","total_2wk_units_mid","total_2wk_units_high"]].copy()

def risk_flag(row):
    inv = row["current_inventory"]
    if inv < row["total_2wk_units_low"]:
        return "🔴 Critical"
    elif inv < row["total_2wk_units_mid"]:
        return "🟠 At risk"
    elif inv < row["total_2wk_units_high"]:
        return "🟡 Watch"
    else:
        return "🟢 OK"

risk["risk_flag"]    = risk.apply(risk_flag, axis=1)
risk["gap_mid"]      = (risk["current_inventory"] - risk["total_2wk_units_mid"]).round(1)
risk["gap_high"]     = (risk["current_inventory"] - risk["total_2wk_units_high"]).round(1)
risk["cover_days_mid"] = (risk["current_inventory"] / (risk["total_2wk_units_mid"] / 14)).round(1)

risk = risk.sort_values(["risk_flag","gap_mid"])

print("=== STOCKOUT RISK ANALYSIS ===")
print(risk[["sku","product_name","tier","current_inventory",
            "total_2wk_units_mid","gap_mid","cover_days_mid","risk_flag"]].to_string(index=False))
print()
print("gap_mid = current_inventory - mid_forecast  (negative = projected stockout)")
print("cover_days_mid = how many days current inventory covers at mid forecast rate")


=== STOCKOUT RISK ANALYSIS ===
       sku             product_name tier  current_inventory  total_2wk_units_mid  gap_mid  cover_days_mid  risk_flag
CYM-MAG-30    Magnesium L-Threonate High                315               704.20  -389.20            6.30 🔴 Critical
CYM-COL-30          Marine Collagen High                211               510.30  -299.30            5.80 🔴 Critical
CYM-VTC-30      Liposomal Vitamin C High                233               525.20  -292.20            6.20 🔴 Critical
CYM-ASH-30 Ashwagandha + L-Theanine High                458               722.30  -264.30            8.90 🔴 Critical
CYM-VTD-30          Vitamin D3 + K2 High                261               467.50  -206.50            7.80 🔴 Critical
CYM-OMG-60              Omega-3 DHA High                218               423.80  -205.80            7.20 🔴 Critical
CYM-PRO-30          Probiotic Daily High                456               650.20  -194.20            9.80 🔴 Critical
CYM-GLU-30    Liposomal Glutathio

## 12. Supply Chain Brief — 3 Bullets

> Ready to send to the Supply Chain team before the promo.


In [17]:
critical = risk[risk["risk_flag"] == "🔴 Critical"].sort_values("gap_mid")
at_risk  = risk[risk["risk_flag"] == "🟠 At risk"]
watch    = risk[risk["risk_flag"] == "🟡 Watch"]

mid_row  = rollup[rollup["scenario"] == "mid"].iloc[0]
low_row  = rollup[rollup["scenario"] == "low"].iloc[0]
high_row = rollup[rollup["scenario"] == "high"].iloc[0]

total_gap_mid  = critical["gap_mid"].sum() + at_risk["gap_mid"].sum()
n_crit         = len(critical)
n_risk         = len(at_risk)
n_total_skus   = len(risk)
post_dip_pct   = (1 - mid_row["total_post_units"] / mid_row["total_promo_units"]) * 100
top3_skus      = mid_contrib.head(3)
top3_str       = ", ".join([
    f"{row['sku']} ({row['contribution_pct']:.0f}%)"
    for _, row in top3_skus.iterrows()
])
top3_pct = mid_contrib["contribution_pct"].head(3).sum()

print("=" * 70)
print("SPRING20 SKU FORECAST — SUPPLY CHAIN BRIEF")
print("=" * 70)
print()

# ── Bullet 1: Volume headline ─────────────────────────────────────────────────
print(
    f"• Mid-case projects {mid_row['total_promo_units']:,.0f} units sold during the "
    f"3-day promo week, followed by {mid_row['total_post_units']:,.0f} units in the "
    f"post-promo week as demand normalises. Total 2-week requirement across all "
    f"in-promo SKUs: {mid_row['total_2wk_units']:,.0f} units "
    f"(range: {low_row['total_2wk_units']:,.0f}–{high_row['total_2wk_units']:,.0f})."
)
print()

# ── Bullet 2: Stockout risks ──────────────────────────────────────────────────
if n_crit == n_total_skus:
    # All SKUs critical — summarise with top 3 worst + aggregate shortfall
    worst3     = critical.head(3)
    worst3_str = "; ".join([
        f"{row['sku']} ({abs(row['gap_mid']):.0f} units short, "
        f"{row['cover_days_mid']:.0f} days cover)"
        for _, row in worst3.iterrows()
    ])
    print(
        f"• ALL {n_crit} in-promo SKUs are CRITICAL — current inventory is below "
        f"the low-case forecast for every SKU. Aggregate mid-case shortfall: "
        f"{abs(total_gap_mid):,.0f} units. The three most exposed: {worst3_str}. "
        f"Immediate action required: expedite reorder on all SKUs prioritised by "
        f"gap size, and cap purchase quantity (e.g. max 2 per customer) to extend "
        f"coverage across the full event window."
    )
elif n_crit > 3:
    # Many critical — top 3 named, rest counted
    worst3     = critical.head(3)
    worst3_str = "; ".join([
        f"{row['sku']} ({abs(row['gap_mid']):.0f} units short)"
        for _, row in worst3.iterrows()
    ])
    print(
        f"• {n_crit} of {n_total_skus} SKUs are CRITICAL (inventory below low-case). "
        f"Top 3 by shortfall: {worst3_str}. "
        f"The remaining {n_crit - 3} are also at risk. "
        f"Expedite reorder across all critical SKUs and cap purchase quantity "
        f"to extend coverage."
    )
elif n_crit > 0:
    crit_str = "; ".join([
        f"{row['sku']} ({abs(row['gap_mid']):.0f} units short)"
        for _, row in critical.iterrows()
    ])
    print(
        f"• {n_crit} SKU(s) are CRITICAL — inventory below low-case: {crit_str}. "
        f"Expedite reorder immediately or cap promo eligibility before launch."
    )
elif n_risk > 0:
    risk_str = "; ".join([
        f"{row['sku']} ({abs(row['gap_mid']):.0f} units short)"
        for _, row in at_risk.iterrows()
    ])
    print(
        f"• {n_risk} SKU(s) are AT RISK at mid-case: {risk_str}. "
        f"Recommend expediting reorder or a 2-unit purchase cap."
    )
else:
    print(
        "• No SKUs are projected to stock out at mid-case. All inventory "
        "levels cover the mid-case 2-week requirement with buffer."
    )
print()

# ── Bullet 3: Post-promo dip + top revenue contributors ──────────────────────
print(
    f"• Post-promo demand is forecast to drop approximately {post_dip_pct:.0f}% "
    f"in the week following the event as pull-forward demand normalises — "
    f"do not replenish to promo-week levels mid-event. "
    f"The top 3 revenue-contributing SKUs are {top3_str}, accounting for "
    f"{top3_pct:.0f}% of total promo revenue — prioritise inventory depth "
    f"and reorder speed on these three first."
)
print()
print("=" * 70)


SPRING20 SKU FORECAST — SUPPLY CHAIN BRIEF

• Mid-case projects 4,206 units sold during the 3-day promo week, followed by 851 units in the post-promo week as demand normalises. Total 2-week requirement across all in-promo SKUs: 5,057 units (range: 4,126–5,802).

• ALL 9 in-promo SKUs are CRITICAL — current inventory is below the low-case forecast for every SKU. Aggregate mid-case shortfall: 2,133 units. The three most exposed: CYM-MAG-30 (389 units short, 6 days cover); CYM-COL-30 (299 units short, 6 days cover); CYM-VTC-30 (292 units short, 6 days cover). Immediate action required: expedite reorder on all SKUs prioritised by gap size, and cap purchase quantity (e.g. max 2 per customer) to extend coverage across the full event window.

• Post-promo demand is forecast to drop approximately 80% in the week following the event as pull-forward demand normalises — do not replenish to promo-week levels mid-event. The top 3 revenue-contributing SKUs are CYM-GLU-30 (18%), CYM-MAG-30 (15%), CYM

## 13. Save Outputs

In [18]:
fc.to_csv("/Users/apple/Desktop/Assignment2/Output/part6_sku_forecast.csv", index=False)
risk.to_csv("/Users/apple/Desktop/Assignment2/Output/part6_stockout_risk.csv", index=False)
print("Saved → part6_sku_forecast.csv")
print("Saved → part6_stockout_risk.csv")
print()
print("=== FINAL MID-CASE SKU FORECAST ===")
print(mid_fc[[
    "sku","product_name","tier","promo_weekly_units",
    "post_weekly_units","total_2wk_units","promo_revenue"
]].sort_values("promo_weekly_units", ascending=False).to_string(index=False))


Saved → part6_sku_forecast.csv
Saved → part6_stockout_risk.csv

=== FINAL MID-CASE SKU FORECAST ===
       sku             product_name tier  promo_weekly_units  post_weekly_units  total_2wk_units  promo_revenue
CYM-ASH-30 Ashwagandha + L-Theanine High              627.80              94.50           722.30      24,608.19
CYM-MAG-30    Magnesium L-Threonate High              606.30              97.90           704.20      30,071.69
CYM-GLU-30    Liposomal Glutathione High              589.80              89.90           679.70      36,801.27
CYM-PRO-30          Probiotic Daily High              561.70              88.50           650.20      29,207.36
CYM-COL-30          Marine Collagen High              421.30              89.10           510.30      22,916.54
CYM-VTC-30      Liposomal Vitamin C High              409.70             115.50           525.20      17,043.35
CYM-VTD-30          Vitamin D3 + K2 High              381.60              85.90           467.50      14,653.90
CYM-